# Health Monitoring API (Live Service - Multi-Pillar Version)
This notebook runs the unified Flask API that integrates all three data pillars:
1. **Pillar A (Wearable):** Vital Signs & Clinical Risk
2. **Pillar B (Behavioral):** Fall Detection & Typing Biomarkers
3. **Pillar C (Environment):** Visual Motion Confirmation

In [ ]:
from flask import Flask, request, jsonify
import numpy as np
import tensorflow as tf
import joblib
import os
from datetime import datetime

app = Flask(__name__)

# --- 1. Load All Trained Models ---
MODELS_DIR = '../../models/'

def load_models():
    try:
        # Pillar A: Health Risk LSTM
        risk_lstm = tf.keras.models.load_model(os.path.join(MODELS_DIR, 'health_risk_v1.h5'))
        # Pillar A: Clinical Risk Classification
        clinical_rf = joblib.load(os.path.join(MODELS_DIR, 'risk_classification_v1.joblib'))
        # Pillar B: Fall Detection
        fall_rf = joblib.load(os.path.join(MODELS_DIR, 'fall_detection_v1.joblib'))
        # Pillar B: Behavioral Anomaly Detector
        behavior_if = joblib.load(os.path.join(MODELS_DIR, 'behavioral_biomarker_v2.joblib'))
        return risk_lstm, clinical_rf, fall_rf, behavior_if
    except Exception as e:
        print(f"Warning: Some models could not be loaded. Ensure training is complete. Error: {e}")
        return None, None, None, None

risk_lstm, clinical_rf, fall_rf, behavior_if = load_models()

@app.route('/predict', methods=['POST'])
def predict():
    try:
        payload = request.json
        # payload structure:
        # { "vitals": { "hr": 72, "spo2": 98, "temp": 36.6 }, 
        #   "behavioral": { "latency": 0.45, "hold_time": 0.12, "phone_acc": [x,y,z] },
        #   "environment": { "camera_motion": bool } }

        results = {}
        alert_level = "Normal"

        # --- Pillar A: Vital Analysis ---
        if clinical_rf and 'vitals' in payload:
            v = payload['vitals']
            vital_features = np.array([[v['hr'], v['spo2'], v['temp']]]) # Simplified for demo
            results['clinical_state'] = clinical_rf.predict(vital_features)[0]

        # --- Pillar B: Behavioral Analysis ---
        if fall_rf and 'behavioral' in payload:
            acc = np.array(payload['behavioral']['phone_acc']).reshape(1, -1)
            fall_detected = bool(fall_rf.predict(acc)[0])
            
            # --- Pillar C: Cross-Check with Camera ---
            if fall_detected:
                camera_sees_motion = payload.get('environment', {}).get('camera_motion', True)
                if not camera_sees_motion:
                    results['fall_status'] = "CONFIRMED FALL - NO MOVEMENT"
                    alert_level = "CRITICAL"
                else:
                    results['fall_status'] = "Possible Fall - Movement Detected"
                    alert_level = "Warning"
            else:
                results['fall_status'] = "No Fall"

        # --- Combined Logic ---
        return jsonify({
            "status": "success",
            "timestamp": datetime.now().isoformat(),
            "predictions": results,
            "alert_level": alert_level
        }), 200

    except Exception as e:
        return jsonify({"status": "error", "message": str(e)}), 400

@app.route('/status', methods=['GET'])
def status():
    return jsonify({"status": "online", "models_loaded": risk_lstm is not None})

if __name__ == '__main__':
    print("Elderly Health AI API starting on port 5000...")
    app.run(host='0.0.0.0', port=5000)